# Reachability-based BLAs for intersection crossings

Companion to [protocol.ipynb](protocol.ipynb) for corridor-valued offers.

- Cells tagged `definitions` are the method.
- Kernel: Python 3.13, `pyspect[hj_reachability]==1.2`.
- Evidence: **[algebra]** normalization · **[semantic]** containment and the fixed client rule ·
  **[numerical]** grid membership.
- Pipeline: contracts → owner obligation $F_s$ → generated query → TLT → HJ realization.
- Scope: point-mass kinematic bicycle on a four-way intersection, exact entry states, nominal HJ-feedback
  client, finite departure library. No robustness certificate.

In [18]:
from pathlib import Path
import sys
import warnings
import numpy as np

warnings.filterwarnings('ignore', message='IProgress not found')  # tqdm without ipywidgets
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src' / 'ltms_reachability').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from scipy.ndimage import distance_transform_edt, maximum_filter
from pyspect import Set
from pyspect.set_builder import BoxSet, Inter, PolytopeSet, Union
from pyspect.impls.hj_reachability import TVHJImpl
from pyspect.systems.hj_reachability import Bicycle4D
from ltms_reachability import LTMSPlanner, TimeWindow, WindowPolicy
from ltms_bla.tlt import compile_query, state_set
from ltms_bla.presentation import table
from ltms_bla.records import implies, ledger_symbols

In [19]:
from fractions import Fraction as F
from math import pi
import interval_logic as il
from protocol import CommitResult, Contract, affine_policies, normalize, validate_commit
from ltms_bla.bargaining import load_bargain
from ltms_bla.checks import aligned_disjoint
from ltms_bla.records import (Admission, Certificate, CorridorBLA, CorridorOffer, CrossingRequest,
                              CrossingService, standing)

# The protocol.ipynb function body; only its offer-eligibility validator differs.
bargain = load_bargain(ROOT)

## 1. Model

Owner controls $r_j(t)\subseteq\mathbb R^2$; client $j$ controls $u=(\delta,a)$.
$$\dot x=v\cos h,\quad \dot y=v\sin h,\quad \dot h=v\tan\delta/L,\quad \dot v=a .$$
Point vehicle at $(x,y)$ with heading $h$ in the global frame.

In [20]:
WHEELBASE = .32                                                                    # L [m]
CONTROLS = dict(min_steer=-pi/4, max_steer=pi/4, min_accel=-.15, max_accel=.15)  # δ [rad], a [m/s²]
SPEED_LIMIT = (.3, .6)                                                             # v [m/s] on every road
VMAX = SPEED_LIMIT[1]

**Map.** A four-way intersection with right-hand traffic, as in the original LTMS.
- Each direction of travel has a lane, entered at one end and left at the other.
- On its lane a vehicle keeps within $\pm\pi/5$ of the lane's heading; in the centre (an octagon) any heading.
- $\mathit{road}_d$ = lane $d$ with its heading band, or the centre. $\mathit{exit}_d$ = exit zone of lane $d$.
- A route enters travelling *inbound* and leaves travelling *outbound* (straight on, or a turn).

In [21]:
# Direction of travel → heading [rad], lane, entry zone and exit zone (boxes over x, y [m]).
MAP = dict(
    east=dict(heading=0, lane=dict(y=(-.8, 0)),
              entry=dict(x=(-2, -1.4), y=(-.68, -.12)), exit=dict(x=(1.4, 2), y=(-.68, -.12))),
    north=dict(heading=pi/2, lane=dict(x=(0, .8)),
               entry=dict(x=(.12, .68), y=(-2, -1.4)), exit=dict(x=(.12, .68), y=(1.4, 2))),
    west=dict(heading=pi, lane=dict(y=(0, .8)),
              entry=dict(x=(1.4, 2), y=(.12, .68)), exit=dict(x=(-2, -1.4), y=(.12, .68))),
    south=dict(heading=-pi/2, lane=dict(x=(-.8, 0)),
               entry=dict(x=(-.68, -.12), y=(1.4, 2)), exit=dict(x=(-.68, -.12), y=(-2, -1.4))),
)
HEADING_BAND = pi/5  # on its lane, a vehicle keeps within ±π/5 of the lane's heading
MANAGED = 1.2        # the centre: the octagon |x| + |y| ≤ 2 within [-1.2, 1.2]²

**Task and capability.** Departure $s$ from `ENTRY[inbound]` ($z(s)=z_0$); lease $[s,s+8]$.
$$\Phi_s:=(\mathit{road}\land\mathit{alloc}_j)\ \mathcal U_{[s,d_\theta]}\ \mathit{exit}_{\rm out}\ \ (\text{read at }s),\qquad
\mathit{road}:=\mathit{road}_{\rm in}\lor\mathit{road}_{\rm out},\qquad T_\theta:=\bigvee_{s\in D_\theta}\Phi_s,$$
$$F_s(r_j):=\exists u(\cdot):\ z^{s,z_0}_u\vDash\Phi_s,\qquad S_a:=\bigvee_{s\in D_\theta}F_s .$$
- $\mathit{alloc}_j$: $p(z(t))\in r_j(t)$, the owner's variable. $F_s$ depends on $r_j$ only: owner-only.
- Client $\pi_r$: realize $V_r$ for $\Phi_s$ with $\mathit{alloc}:=r$; execute $u\in\arg\min_u\nabla V_r\cdot f(z,u)$.
  One rule, parameterized by the offered allocation.
- Departure: earliest $s$ whose $F_s$ the agreement implies.

In [22]:
# (x, y, h, v) at departure: the middle of each entry zone, heading along the lane.
ENTRY = dict(east=(-1.6, -.4, 0, .45), north=(.4, -1.6, pi/2, .45),
             west=(1.6, .4, pi, .45), south=(-.4, 1.6, -pi/2, .45))
LEASE = 8  # lease on [s, s+8]


def certified_window_applies(departure, deadline):
    # The certifying query uses the lease window [s, s+8] ⊆ [s, d]; then it implies F_s.
    return departure + LEASE <= deadline

| | Meaning | Actor | Evidence |
|:--|:--|:--|:--|
| $H_\kappa$ | $r_j\supseteq R_\kappa$ on $[s,s+8]$ | owner | promise; $R_\kappa$ from §2 |
| $F_s$ | $\exists u$: $\Phi_s$ under $r_j$ | owner | query generated from $F_s$ with $\mathit{alloc}:=R_\kappa$ (§2) |
| $S_a$ | $\bigvee F_s$ | owner | definition |
| $\Phi_s$ | $(\mathit{road}\land\mathit{alloc}_j)\,\mathcal U_{[s,d]}\,\mathit{exit}$ | client | $\pi_r$ given $F_s(r)$ |

**[semantic]** $F_s(r)$ and $\pi_r$ give $\Phi_s$ (nominal dynamics, exact $V_r$), so $C_{\rm client}=\langle S_a\mid T_\theta\rangle$.
$\mathit{alloc}$ occurs only positively in $\Phi_s$, so $F_s$ is monotone in $r$: $H_\kappa\vDash_f F_s\iff F_s(R_\kappa)$.

## 2. Environment, corridor and query

Grid over $(t,x,y,h,v)$: 9 s in 0.25 s steps. The heading axis is periodic and covers one turn from $-3\pi/4$,
so no lane's heading band wraps around. §1's map becomes pyspect set builders (inside ⇔ $\le0$).

In [23]:
HORIZON, DT = 9, .25
AXES = [dict(name='t', bounds=[0, HORIZON], step=DT, unit='s'),
        dict(name='x', bounds=[-2, 2], points=41, unit='m'), dict(name='y', bounds=[-2, 2], points=41, unit='m'),
        dict(name='*h', bounds=[-3*pi/4, 5*pi/4], points=16, unit='rad'),  # '*': periodic
        dict(name='v', bounds=list(SPEED_LIMIT), points=5, unit='m/s')]
impl = TVHJImpl(dict(cls=Bicycle4D, wheelbase=WHEELBASE, **CONTROLS), AXES, accuracy='medium')
x, y, h, v = (np.asarray(c) for c in impl.grid.coordinate_vectors)


def box(**bounds):
    return BoxSet(bounds, axes=list(bounds))


OCTAGON = PolytopeSet(normals=[[1, 1], [-1, 1], [-1, -1], [1, -1]],  # |x| + |y| ≤ 2
                      offsets=[[-1, -1], [1, -1], [1, 1], [-1, 1]], axes=['x', 'y'])
SPEED = box(v=SPEED_LIMIT)
CENTRE = Inter(Inter(OCTAGON, box(x=(-MANAGED, MANAGED), y=(-MANAGED, MANAGED))), SPEED)  # any heading
ROAD = {d: Union(Inter(box(**m['lane'], h=(m['heading'] - HEADING_BAND, m['heading'] + HEADING_BAND)), SPEED),
                 CENTRE) for d, m in MAP.items()}
ENTRY_ZONE = {d: Inter(box(**m['entry']), ROAD[d]) for d, m in MAP.items()}
EXIT_ZONE = {d: Inter(box(**m['exit']), ROAD[d]) for d, m in MAP.items()}
INTERPRETATION = {f'road_{d[0]}': ROAD[d] for d in MAP} | {f'exit_{d[0]}': EXIT_ZONE[d] for d in MAP}

In [ ]:
# Roads (grey), entry zones (blue) and exit zones (red), projected over (t, h, v); arrows at the entry states.
figure = impl.plot(*((ROAD[d](impl), dict(colorscale='Greys', value=.3)) for d in MAP),
                   *((ENTRY_ZONE[d](impl), dict(colorscale='Blues', value=.6)) for d in MAP),
                   *((EXIT_ZONE[d](impl), dict(colorscale='Reds', value=.6)) for d in MAP),
                   method='bitmap', axes=('x', 'y'),
                   layout=dict(title='Four-way intersection', width=560, height=540))
for d, (px, py, ph, _) in ENTRY.items():
    figure.add_annotation(x=px + .5*np.cos(ph), y=py + .5*np.sin(ph), ax=px, ay=py, axref='x', ayref='y',
                          text=d, arrowhead=2, arrowwidth=2)
figure.show()

## 3. Contracts, admission and offers

$$C_{\rm client}=\langle S_a\mid T_\theta\rangle,\quad R_{\rm task}=\langle\top\mid T_\theta\rangle,$$
$$Q=R_{\rm task}/C_{\rm client}=\langle\neg S_a\lor T_\theta\mid S_a\rangle,\quad
C_{\rm cmt}^{+}=C_{\rm cmt}\wedge Q=\langle\top\mid L\land S_a\rangle .$$
- **Feasible departures** $F\subseteq D_\theta$: $s+8\le d_\theta$, the route's generated query holds, and its
  reservation shifted to $s$ is disjoint from every standing one. Together these witness $L\land S_a$.
- Try the original service, then permitted alternatives **in order**; rebuild each quotient.
  Exhaustion returns `None` (`NO-BLA`).

In [ ]:
def service_contracts(service):
    # III-A: the fixed client and the joint task for one selected service.
    S_a, T = il.Or(service.access_for(s) for s in service.departures), service.task
    return S_a, T, Contract(S_a, T), Contract(True, T)


def feasible(service, departure, ledger, base):
    # Capability with its window, and compatibility with every standing reservation.
    routes = base['routes']
    return (routes[service.route]['capability'] and certified_window_applies(departure, service.deadline)
            and all(aligned_disjoint(routes[service.route]['reserved'], departure,
                                     routes[b.offer.service.route]['reserved'], b.offer.departure, base['dt'])
                    for b in ledger))


def admit(request, ledger, base):
    # Frozen snapshot: the standing commitments C_cmt = <T | L> stay fixed for this attempt.
    ledger = tuple(ledger)
    commitment = standing(ledger)

    # III-B, Service selection: original service, then permitted alternatives.
    for service in request.services:
        # III-B, Requested task: instantiate the client and joint target.
        S_a, T, C_client, R_task = service_contracts(service)

        # III-B, Owner obligations: derive Q and check its access-only guarantee.
        Q = R_task.quotient(C_client).normalize()
        assert Q.assumption == normalize(~S_a | T)
        assert Q.guarantee == normalize(S_a)

        # III-B, Admission output: retain standing obligations, C_plus = C_cmt ∧ Q.
        C_plus = commitment.conjoin(Q)
        assert normalize(C_plus.guarantee) == normalize(commitment.guarantee & S_a)

        # III-B, Admission output: the feasible departures of the finite family.
        departures = tuple(s for s in service.departures if feasible(service, s, ledger, base))
        if departures:
            return Admission(request, service, S_a, C_client, R_task, Q, C_plus, departures, commitment)
    # III-B, Service selection: exhaustion gives NO-BLA; commitments are unchanged.
    return None

**Offers.** $K_\kappa=\langle\top\mid H_\kappa\land F_s\rangle$ for each $s\in F$.
- $K_\kappa\Rightarrow S_a$ is algebraic; that $F_s$ is redundant given $H_\kappa$ is the capability evidence.
- Compatibility: disjoint reservations, so $r_i:=R_{\kappa_i}$ satisfies $L\land H_\kappa$; inherited from admission.
- **[algebra]** $K_\kappa\preceq Q$, $C_{\rm cmt}\wedge K_\kappa\preceq C_{\rm cmt}^{+}$,
  $C_{\rm client}\otimes(C_{\rm cmt}\wedge K_\kappa)\preceq C_{\rm cmt}\wedge R_{\rm task}$.

In [ ]:
def generate(admission, base):
    offers = []
    # III-C, Compatible instances: one offer per feasible departure of the selected service.
    for s in (() if admission is None else admission.feasible_departures):
        # Record the evidence; compatibility follows from s in F, without another comparison.
        evidence = Certificate(capability=base['routes'][admission.service.route]['capability'],
                               window_applies=certified_window_applies(s, admission.service.deadline),
                               compatible=s in admission.feasible_departures)
        offers.append(CorridorOffer(admission.service, admission.access, s, F(s + LEASE), evidence))
    return tuple(offers)

**Bargaining and commitment.** `bargain` is verbatim from `protocol.ipynb`; its validator requires
certified compatible offers with $K_\kappa\Rightarrow S_a$, all of one service. Commit stores the exact accepted
pair, $C_{\rm cmt}\leftarrow C_{\rm cmt}\wedge K^*$; `NO-BLA` preserves commitments and ledger.

In [ ]:
def commit(admission, offers, negotiation, commitment, ledger=()):
    ledger = tuple(ledger)
    if negotiation.status == 'NO-BLA' and negotiation.agreement is None:
        return CommitResult('NO-BLA', commitment, ledger, None)
    # Check the frozen commitment/offers and exact accepted pair; do not regenerate.
    validate_commit(admission, offers, negotiation, commitment)
    record = CorridorBLA(negotiation.agreement.offer, negotiation.agreement.price)
    return CommitResult('COMMITTED', commitment.conjoin(record.contract), (*ledger, record), record)


def attempt(spec, ledger, base):
    # admit → generate → bargain → commit for one request against the frozen ledger.
    ledger = tuple(ledger)
    admission = admit(spec['request'], ledger, base)
    offers = generate(admission, base)
    policies = affine_policies(dict(zip(offers, spec['costs'])), dict(zip(offers, spec['values'])))
    negotiation = bargain(offers, policies, spec['targets'], spec['minima'], PRICES, ROUND_BUDGET)
    result = commit(admission, offers, negotiation, standing(ledger), ledger)
    return dict(request=spec['request'], admission=admission, offers=offers, negotiation=negotiation,
                result=result)

## 4. Staged execution

| Client | Route | Case |
|:--|:--|:--|
| A | east | first crossing |
| B | west | two alternatives during A's lease, in the opposite lane |
| C | north | crosses A and B in the centre: no feasible departure |
| D | south | price failure |
| E | east → north | left turn; original deadline: window does not fit; permitted later service |

Costs/values: earlier offer 5/10, later 2/8; D values 1/0.

In [ ]:
PRICES, ROUND_BUDGET = (F(0), F(20)), 3
PREFERENCES = dict(costs=(F(5), F(2)), values=(F(10), F(8)),
                   targets={'owner': F(4), 'client': F(4)}, minima={'owner': F(1), 'client': F(1)})
PRICE_FAILURE = dict(costs=(F(5), F(2)), values=(F(1), F(0)),
                     targets={'owner': F(4), 'client': F(1)}, minima={'owner': F(2), 'client': F(0)})
REQUESTS = (
    dict(request=CrossingRequest('A', CrossingService('A', 'east', 'east', (0,), 8)), **PREFERENCES),
    dict(request=CrossingRequest('B', CrossingService('B', 'west', 'west', (0, 1), 9)), **PREFERENCES),
    dict(request=CrossingRequest('C', CrossingService('C', 'north', 'north', (0, 1), 9)), **PREFERENCES),
    dict(request=CrossingRequest('D', CrossingService('D', 'south', 'south', (10, 11), 19)), **PRICE_FAILURE),
    dict(request=CrossingRequest('E', CrossingService('E', 'east', 'north', (20,), 22),
                                 alternatives=(CrossingService('E', 'east', 'north', (20,), 28),)), **PREFERENCES),
)

In [ ]:
# One reservation and generated query per route the requests use.
routes = sorted({s.route for spec in REQUESTS for s in spec['request'].services})
BASE = dict(dt=DT, routes={r: EAST if r == EAST['template'].route else route_evidence(*r) for r in routes})
table(['Route', 'Reserved cells', '$V_R$ at the entry state', 'Capability'],
      [(' → '.join(r), int(e['reserved'].sum()), f"{e['value']:.5f}", e['capability'])
       for r, e in BASE['routes'].items()])


Running for 36 iterations:   0%|          | 0/36 [00:00<?, ?it/s]


Running for 36 iterations:   8%|▊         | 3/36 [00:00<00:01, 23.77it/s]


Running for 36 iterations:  17%|█▋        | 6/36 [00:00<00:01, 23.65it/s]


Running for 36 iterations:  25%|██▌       | 9/36 [00:00<00:01, 23.07it/s]


Running for 36 iterations:  33%|███▎      | 12/36 [00:00<00:01, 23.53it/s]


Running for 36 iterations:  42%|████▏     | 15/36 [00:00<00:00, 24.20it/s]

... (43 outputs left out)


Running for 36 iterations:  83%|████████▎ | 30/36 [00:01<00:00, 23.48it/s]


Running for 36 iterations:  92%|█████████▏| 33/36 [00:01<00:00, 23.47it/s]


Running for 36 iterations: 100%|██████████| 36/36 [00:01<00:00, 24.35it/s]

| Route | Reserved cells | $V_R$ at the entry state | Capability |
| --- | --- | --- | --- |
| east → east | 5312 | -0.10215 | True |
| east → north | 13338 | -0.10183 | True |
| north → north | 5312 | -0.10215 | True |
| south → south | 5312 | -0.10215 | True |
| west → west | 5313 | -0.10217 | True |

### B: admission and offers against A

In [ ]:
attempts = [attempt(REQUESTS[0], (), BASE)]  # A
ledger = attempts[-1]['result'].ledger
B_spec = REQUESTS[1]
admission = admit(B_spec['request'], ledger, BASE)
offers = generate(admission, BASE)
assert admission.feasible_departures == (0, 1) and len(offers) == 2
symbols = ledger_symbols(ledger, *offers)
label = lambda o: f'κ(s={o.departure})'
table(['Object [algebra]', 'Value'], [
    ('$S_a$', admission.access), (r'$C_{\rm client}$', admission.client_contract),
    (r'$R_{\rm task}$', admission.task), ('$Q$', admission.quotient),
    (r'$C_{\rm cmt}$ (A)', admission.commitment), (r'$C_{\rm cmt}^{+}$', admission.obligations),
    ('Feasible departures $F$', admission.feasible_departures)], symbols)
table(['Offer', 'Lease [s]', '$K_\\kappa$', 'Query: $z_0$ inside', 'Window ⊆ [s, d]',
       'Compatible (from admission)', '$K_\\kappa\\Rightarrow S_a$ [algebra]'],
      [(label(o), f'[{o.departure}, {o.end}]', o.contract, *vars(o.certificate).values(),
        implies(o.contract.guarantee, admission.access)) for o in offers], symbols)

| Object [algebra] | Value |
| --- | --- |
| $S_a$ | $F^{B}_{0} \lor F^{B}_{1}$ |
| $C_{\rm client}$ | $\displaystyle \left\langle F^{B}_{0} \lor F^{B}_{1} \;\middle\vert\; \neg F^{B}_{0} \land \neg F^{B}_{1} \lor \left(r_{B} \land \mathit{road}_w\right) \mathbin{\mathrm{U}_{[0,9]}} \mathit{exit}_w \lor \left(r_{B} \land \mathit{road}_w\right) \mathbin{\mathrm{U}_{[1,9]}} \mathit{exit}_w \right\rangle$ |
| $R_{\rm task}$ | $\displaystyle \left\langle \top \;\middle\vert\; \left(r_{B} \land \mathit{road}_w\right) \mathbin{\mathrm{U}_{[0,9]}} \mathit{exit}_w \lor \left(r_{B} \land \mathit{road}_w\right) \mathbin{\mathrm{U}_{[1,9]}} \mathit{exit}_w \right\rangle$ |
| $Q$ | $\displaystyle \left\langle \left(\neg F^{B}_{0} \lor \left(r_{B} \land \mathit{road}_w\right) \mathbin{\mathrm{U}_{[0,9]}} \mathit{exit}_w \lor \left(r_{B} \land \mathit{road}_w\right) \mathbin{\mathrm{U}_{[1,9]}} \mathit{exit}_w\right) \land \left(\neg F^{B}_{1} \lor \left(r_{B} \land \mathit{road}_w\right) \mathbin{\mathrm{U}_{[0,9]}} \mathit{exit}_w \lor \left(r_{B} \land \mathit{road}_w\right) \mathbin{\mathrm{U}_{[1,9]}} \mathit{exit}_w\right) \;\middle\vert\; F^{B}_{0} \lor F^{B}_{1} \right\rangle$ |
| $C_{\rm cmt}$ (A) | $\displaystyle \left\langle \top \;\middle\vert\; F^{A}_{0} \land H^{A}_{0} \right\rangle$ |
| $C_{\rm cmt}^{+}$ | $\displaystyle \left\langle \top \;\middle\vert\; F^{A}_{0} \land H^{A}_{0} \land \left(F^{B}_{0} \lor F^{B}_{1}\right) \right\rangle$ |
| Feasible departures $F$ | (0, 1) |

| Offer | Lease [s] | $K_\kappa$ | Query: $z_0$ inside | Window ⊆ [s, d] | Compatible (from admission) | $K_\kappa\Rightarrow S_a$ [algebra] |
| --- | --- | --- | --- | --- | --- | --- |
| κ(s=0) | [0, 8] | $\displaystyle \left\langle \top \;\middle\vert\; F^{B}_{0} \land H^{B}_{0} \right\rangle$ | True | True | True | True |
| κ(s=1) | [1, 9] | $\displaystyle \left\langle \top \;\middle\vert\; F^{B}_{1} \land H^{B}_{1} \right\rangle$ | True | True | True | True |

### B: bargaining and commitment

In [ ]:
policies = affine_policies(dict(zip(offers, B_spec['costs'])), dict(zip(offers, B_spec['values'])))
negotiation = bargain(offers, policies, B_spec['targets'], B_spec['minima'], PRICES, ROUND_BUDGET)
result = commit(admission, offers, negotiation, admission.commitment, ledger)
service = admission.service
# Fixed client rule: the earliest s whose F_s the commitment implies.
departure = min(s for s in service.departures if implies(result.commitment.guarantee, service.access_for(s)))
table(['Round', 'Proposer', 'Targets (owner, client)', 'Offer', 'Price', 'Utilities (owner, client)', 'Accepted'],
      [(r.number, r.proposer, tuple(v for _, v in r.targets), label(r.proposal.offer), r.proposal.price,
        tuple(v for _, v in r.utilities), r.accepted) for r in negotiation.history])
table(['Commitment', 'Value'], [
    ('Status', result.status), ('$C_{\\rm cmt}$ after B', result.commitment),
    ('Departure chosen by the fixed client rule', departure)], ledger_symbols(result.ledger))
assert result.status == 'COMMITTED' and departure == result.agreement.offer.departure
attempts.append(dict(request=B_spec['request'], admission=admission, offers=offers,
                     negotiation=negotiation, result=result))
ledger = result.ledger

| Round | Proposer | Targets (owner, client) | Offer | Price | Utilities (owner, client) | Accepted |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | owner | (4, 4) | κ(s=0) | 9 | (4, 1) | False |
| 2 | client | (5/2, 5/2) | κ(s=0) | 15/2 | (5/2, 5/2) | True |

| Commitment | Value |
| --- | --- |
| Status | COMMITTED |
| $C_{\rm cmt}$ after B | $\displaystyle \left\langle \top \;\middle\vert\; F^{A}_{0} \land F^{B}_{0} \land H^{A}_{0} \land H^{B}_{0} \right\rangle$ |
| Departure chosen by the fixed client rule | 0 |

### C–E and summary

In [ ]:
for spec in REQUESTS[2:]:
    record = attempt(spec, ledger, BASE)
    attempts.append(record)
    ledger = record['result'].ledger
table(['Client', 'Selected service', 'Feasible departures $F$', 'Offers', 'Rounds', 'Outcome', 'Price'],
      [(a['request'].client, '—' if a['admission'] is None else f"d={a['admission'].service.deadline}",
        '—' if a['admission'] is None else a['admission'].feasible_departures, len(a['offers']),
        len(a['negotiation'].history), a['result'].status,
        '—' if a['result'].agreement is None else a['result'].agreement.price) for a in attempts])
assert [a['result'].status for a in attempts] == ['COMMITTED', 'COMMITTED', 'NO-BLA', 'NO-BLA', 'COMMITTED']
assert [b.offer.service.client for b in ledger] == ['A', 'B', 'E']
assert attempts[4]['admission'].service.deadline == 28

| Client | Selected service | Feasible departures $F$ | Offers | Rounds | Outcome | Price |
| --- | --- | --- | --- | --- | --- | --- |
| A | d=8 | (0) | 1 | 2 | COMMITTED | 15/2 |
| B | d=9 | (0, 1) | 2 | 2 | COMMITTED | 15/2 |
| C | — | — | 0 | 0 | NO-BLA | — |
| D | d=19 | (10, 11) | 2 | 3 | NO-BLA | — |
| E | d=28 | (20) | 1 | 2 | COMMITTED | 15/2 |

## Reproduction

```sh
uv run --group notebooks python scripts/run_ltms_notebook.py --save
```